# CSCAS baseline comparison

Compares three baselines evaluated on the identical CSCAS train/test split and training-pool protocol (see `baselines/cscas.py`, `baselines/cscas_base.py`, `baselines/cscas_bert.py`):

1. **Paper** -- the CSCAS paper's own 42 raw features, `RandomForestClassifier`
2. **Base schema** -- this project's 41-feature `base` schema (paper's features minus the raw `SignatureID` identifier), `RandomForestClassifier`
3. **BERT** -- 8 non-similarity fields serialized to text, fine-tuned DistilBERT

All three share the same temporal train/test split, the same two training pools (Baseline 1: random undersampling; Baseline 2: guided by CSCAS's SCAS outlier clusters), and the same 5-seed averaging -- only the model/feature representation differs.

**Run the three scripts first** to generate the `results/*.json` files this notebook reads:
```
cd src/thesis/baselines
python cscas.py
python cscas_base.py
python cscas_bert.py
```

In [1]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis.baselines._results import load_baseline_results

## Settings

Edit and re-run -- nothing past this cell needs to change.

In [2]:
RESULT_NAMES = ["cscas", "cscas_base", "cscas_bert"]

LABELS = {
    "cscas": "Paper (42 features, RF)",
    "cscas_base": "Base schema (41 features, RF)",
    "cscas_bert": "BERT (8 fields as text)",
}

COLORS = {
    "cscas": "#4477AA",
    "cscas_base": "#CCBB44",
    "cscas_bert": "#EE6677",
}

In [3]:
results = {}
for name in RESULT_NAMES:
    try:
        results[name] = load_baseline_results(name)
    except FileNotFoundError as e:
        print(f"[skip] {e}")

print(f"Loaded results for: {list(results.keys())}")

[skip] /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/cscas.json not found -- run `python cscas.py` from src/thesis/baselines/ first.
[skip] /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/cscas_base.json not found -- run `python cscas_base.py` from src/thesis/baselines/ first.
[skip] /Users/annavisman/stack/TUDelft/thesis/msc-thesis/src/thesis/baselines/results/cscas_bert.json not found -- run `python cscas_bert.py` from src/thesis/baselines/ first.
Loaded results for: []


In [4]:
def plot_baseline(baseline_key: str, title: str) -> None:
    metrics = ["precision", "recall", "f1"]
    methods = [n for n in RESULT_NAMES if n in results]
    if not methods:
        print("No results loaded -- run the baseline scripts first.")
        return

    x = np.arange(len(metrics))
    width = 0.8 / len(methods)

    fig, ax = plt.subplots(figsize=(7, 4))
    for i, name in enumerate(methods):
        values = [results[name][baseline_key][m] for m in metrics]
        offset = (i - (len(methods) - 1) / 2) * width
        bars = ax.bar(
            x + offset, values, width,
            label=LABELS.get(name, name), color=COLORS.get(name),
        )
        ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)

    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.show()

## Baseline 1: random undersampling

In [5]:
plot_baseline("baseline_1", "Baseline 1: random undersampling")

No results loaded -- run the baseline scripts first.


## Baseline 2: guided by CSCAS

In [6]:
plot_baseline("baseline_2", "Baseline 2: guided by CSCAS")

No results loaded -- run the baseline scripts first.


## Summary table

Tidy view of all loaded results -- handy to copy straight into a slide.

In [ ]:
BASELINE_LABELS = {"baseline_1": "Random undersampling", "baseline_2": "Guided by CSCAS"}

rows = []
for name, data in results.items():
    for baseline_key, baseline_label in BASELINE_LABELS.items():
        rows.append({
            "method": LABELS.get(name, name),
            "baseline": baseline_label,
            "precision": data[baseline_key]["precision"],
            "recall": data[baseline_key]["recall"],
            "f1": data[baseline_key]["f1"],
        })

if not rows:
    print("No results loaded -- run the baseline scripts first.")
    summary_df = pd.DataFrame(columns=["method", "baseline", "precision", "recall", "f1"])
else:
    summary_df = pd.DataFrame(rows).sort_values(["baseline", "method"]).reset_index(drop=True)
summary_df